In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
df_silver = spark.read.table("etl_project.silver.customers")

display(df_silver)

customer_id,email,city,state,domains,full_name
C00001,rushjeff@ryan.org,Johnsonmouth,MS,ryan.org,Emily Mooney
C00002,mccoykiara@kelly.com,Stephenfort,WY,kelly.com,Andrea Sellers
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,yahoo.com,Craig Hayes
C00004,lawrence05@campbell.info,Chrisland,ND,campbell.info,Bryan Scott
C00005,carrie45@yahoo.com,East Dennistown,RI,yahoo.com,Sean Vasquez
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy
C00007,scottallen@gmail.com,Joneshaven,VA,gmail.com,Amanda Doyle
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,horton-adams.com,Paul Campos
C00009,dennis03@yahoo.com,Kimberlyview,MD,yahoo.com,Mary Green
C00010,charles58@murillo.net,West Hector,OK,murillo.net,James Myers


In [0]:
df_silver = df_silver.dropDuplicates(["customer_id"])

In [0]:
from pyspark.sql.functions import current_timestamp

df_silver = df_silver.withColumn("create_date", current_timestamp()) \
                     .withColumn("update_date", current_timestamp())

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, lit

df_silver = df_silver.withColumn(
    "DimCustomerKey",
    monotonically_increasing_id() + lit(1)
)

In [0]:
display(df_silver)

customer_id,email,city,state,domains,full_name,create_date,update_date,DimCustomerKey
C00001,rushjeff@ryan.org,Johnsonmouth,MS,ryan.org,Emily Mooney,2026-03-07T20:43:57.943Z,2026-03-07T20:43:57.943Z,1
C00002,mccoykiara@kelly.com,Stephenfort,WY,kelly.com,Andrea Sellers,2026-03-07T20:43:57.943Z,2026-03-07T20:43:57.943Z,2
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,yahoo.com,Craig Hayes,2026-03-07T20:43:57.943Z,2026-03-07T20:43:57.943Z,3
C00004,lawrence05@campbell.info,Chrisland,ND,campbell.info,Bryan Scott,2026-03-07T20:43:57.943Z,2026-03-07T20:43:57.943Z,4
C00005,carrie45@yahoo.com,East Dennistown,RI,yahoo.com,Sean Vasquez,2026-03-07T20:43:57.943Z,2026-03-07T20:43:57.943Z,5
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy,2026-03-07T20:43:57.943Z,2026-03-07T20:43:57.943Z,6
C00007,scottallen@gmail.com,Joneshaven,VA,gmail.com,Amanda Doyle,2026-03-07T20:43:57.943Z,2026-03-07T20:43:57.943Z,7
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,horton-adams.com,Paul Campos,2026-03-07T20:43:57.943Z,2026-03-07T20:43:57.943Z,8
C00009,dennis03@yahoo.com,Kimberlyview,MD,yahoo.com,Mary Green,2026-03-07T20:43:57.943Z,2026-03-07T20:43:57.943Z,9
C00010,charles58@murillo.net,West Hector,OK,murillo.net,James Myers,2026-03-07T20:43:57.943Z,2026-03-07T20:43:57.943Z,10


In [0]:
table_exists = spark.catalog.tableExists("etl_project.gold.DimCustomers")

print(table_exists)

False


In [0]:
from delta.tables import DeltaTable

if not table_exists:

    df_silver.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("etl_project.gold.DimCustomers")

else:

    gold_table = DeltaTable.forName(spark, "etl_project.gold.DimCustomers")

    (
    gold_table.alias("t")
    .merge(
        df_silver.alias("s"),
        "t.customer_id = s.customer_id"
    )
    .whenMatchedUpdate(
        set={
            "email": "s.email",
            "city": "s.city",
            "state": "s.state",
            "domains": "s.domains",
            "full_name": "s.full_name",
            "update_date": "s.update_date"
        }
    )
    .whenNotMatchedInsert(
        values={
            "DimCustomerKey": "s.DimCustomerKey",
            "customer_id": "s.customer_id",
            "email": "s.email",
            "city": "s.city",
            "state": "s.state",
            "domains": "s.domains",
            "full_name": "s.full_name",
            "create_date": "s.create_date",
            "update_date": "s.update_date"
        }
    )
    .execute()
    )

In [0]:
df_gold = spark.read.table("etl_project.gold.DimCustomers")

display(df_gold)

customer_id,email,city,state,domains,full_name,create_date,update_date,DimCustomerKey
C00010,charles58@murillo.net,West Hector,OK,murillo.net,James Myers,2026-03-07T20:46:30.988Z,2026-03-07T20:46:30.988Z,1
C00045,dkhan@hotmail.com,North Kara,OK,hotmail.com,Matthew Lee,2026-03-07T20:46:30.988Z,2026-03-07T20:46:30.988Z,2
C00049,kristinawalsh@gmail.com,New Heatherside,IA,gmail.com,Mike Harvey,2026-03-07T20:46:30.988Z,2026-03-07T20:46:30.988Z,3
C00056,cruzkristen@hotmail.com,Catherineberg,WA,hotmail.com,Tina Cantu,2026-03-07T20:46:30.988Z,2026-03-07T20:46:30.988Z,4
C00062,kimdennis@yahoo.com,Kaitlynburgh,MI,yahoo.com,Shelley Holland,2026-03-07T20:46:30.988Z,2026-03-07T20:46:30.988Z,5
C00094,rogersrandall@gonzales.org,Harrisonmouth,AK,gonzales.org,Hannah Olson,2026-03-07T20:46:30.988Z,2026-03-07T20:46:30.988Z,6
C00104,gloria15@yahoo.com,New Danielle,VA,yahoo.com,Patrick Meadows,2026-03-07T20:46:30.988Z,2026-03-07T20:46:30.988Z,7
C00105,christine52@kennedy.com,Crystalmouth,AL,kennedy.com,April Wright,2026-03-07T20:46:30.988Z,2026-03-07T20:46:30.988Z,8
C00126,bonniewhite@smith.com,Gregoryton,WA,smith.com,Amanda King,2026-03-07T20:46:30.988Z,2026-03-07T20:46:30.988Z,9
C00152,tara25@guzman-nelson.net,Ashleyside,ID,guzman-nelson.net,David Salazar,2026-03-07T20:46:30.988Z,2026-03-07T20:46:30.988Z,10


In [0]:
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://golddb@akashdeltalake.dfs.core.windows.net/DimCustomers")